# SciVer Qdrant Basic Usage

This notebook demonstrates how to use the local SciVer Qdrant vector database as a reusable feature store. It shows how to connect to the database, inspect collection metadata, browse chart-claim pair records, run nearest-neighbor searches, apply metadata filters, and retrieve a point by deterministic ID.

## 1. Setup

The code below resolves the repository root whether the notebook is executed from the repo root or from inside `notebooks/`. The editable variables at the bottom identify the local Qdrant path and the main supervised pair collection.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from qdrant_client import QdrantClient, models

# Resolve the repo root robustly for both `jupyter notebook notebooks/...` and nbconvert from repo root.
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "sciver_vector_db").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

from sciver_vector_db.qdrant_store import point_uuid, query_points

# Prefer the fetch-first processed location; fall back to the legacy path if it exists.
QDRANT_PATH = REPO_ROOT / "data/processed/qdrant_sciver"
if not QDRANT_PATH.exists() and (REPO_ROOT / ".qdrant_sciver").exists():
    QDRANT_PATH = REPO_ROOT / ".qdrant_sciver"
COLLECTION = "sciver_pairs"

print(f"Repo root: {REPO_ROOT}")
print(f"Qdrant path: {QDRANT_PATH}")
print(f"Collection: {COLLECTION}")

Repo root: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase
Qdrant path: /Users/conglongxu/Projects/Erdos-2016-Summer/codebase/data/processed/qdrant_sciver
Collection: sciver_pairs


## 2. Connect and Inspect the Collection

Qdrant stores one point per usable chart/table-claim pair in `sciver_pairs`. Each point has metadata in `payload` and multiple named vectors such as `claim_vec`, `pair_text_vec`, and `image_vec`.

In [2]:
if not QDRANT_PATH.exists():
    raise FileNotFoundError(f"Qdrant database not found at {QDRANT_PATH}. Build it before running this notebook.")

# Local mode opens the embedded Qdrant store directly from disk.
client = QdrantClient(path=str(QDRANT_PATH))
collection_info = client.get_collection(COLLECTION)

# Named vector metadata lives under collection_info.config.params.vectors.
vectors_config = collection_info.config.params.vectors
vector_rows = []
for name, params in vectors_config.items():
    vector_rows.append({"vector_name": name, "size": params.size, "distance": str(params.distance)})

print(f"Point count: {collection_info.points_count}")
pd.DataFrame(vector_rows)

Point count: 1500


,vector_name,size,distance
0,claim_vec,384,Cosine
1,evidence_text_vec,384,Cosine
2,pair_text_vec,384,Cosine
3,image_vec,512,Cosine


## 3. Browse Sample Pair Records

Scrolling returns records without ranking. Here we inspect payload fields, which are the human-readable metadata and supervised labels. Vectors are available separately and are not printed in full because they are high-dimensional arrays.

In [3]:
# Fetch a few records with vectors so one can be reused as a query seed later.
points, _ = client.scroll(
    collection_name=COLLECTION,
    limit=5,
    with_payload=True,
    with_vectors=True,
)
if not points:
    raise RuntimeError(f"No points found in {COLLECTION}.")

sample_rows = []
for point in points:
    payload = point.payload or {}
    sample_rows.append(
        {
            "pair_id": payload.get("pair_id"),
            "split": payload.get("split"),
            "label": payload.get("label"),
            "claim_type": payload.get("claim_type"),
            "modality": payload.get("modality"),
            "claim": payload.get("claim"),
            "image_path": payload.get("image_path"),
        }
    )

pd.DataFrame(sample_rows)

,pair_id,split,label,claim_type,modality,claim,image_path
0,pair:val:2410.01727v1:631:e591e8dfb5724723:231...,val,entailed,direct,table,"On the Eedi dataset, sparseKT model’s AUC incr...",/Users/conglongxu/Projects/Erdos-2016-Summer/d...
1,pair:test:2410.21526v1:641:4bc58bc2b03b9c0e:a2...,test,refuted,analytical,table,Since the MRPC synthetic set contains 3005 sam...,/Users/conglongxu/Projects/Erdos-2016-Summer/d...
2,pair:val:2410.22046v2:396:5fba8c88a57dc187:73d...,val,entailed,analytical,chart,"The prominence of G (13.8%), C (11.4%), and D ...",/Users/conglongxu/Projects/Erdos-2016-Summer/d...
3,pair:test:2410.08387v1:453:29ac45dce1a30b46:82...,test,refuted,analytical,table,"Under the optically thin, single-temperature m...",/Users/conglongxu/Projects/Erdos-2016-Summer/d...
4,pair:val:2409.00140v1:475:2ce2ad80db230f92:cdb...,val,refuted,analytical,table,Pooled data shows the FQReLU-QIP-1-2 variant a...,/Users/conglongxu/Projects/Erdos-2016-Summer/d...


## 4. Search Nearest Neighbors

A seed record can be searched against different named vectors. `pair_text_vec` searches by combined claim and evidence text, `claim_vec` searches by claim semantics only, and `image_vec` searches by chart image embedding when image vectors were built.

In [4]:
seed = points[0]
seed_payload = seed.payload or {}
seed_vectors = seed.vector or {}

print("Seed record")
print(f"  pair_id: {seed_payload.get('pair_id')}")
print(f"  label:   {seed_payload.get('label')}")
print(f"  claim:   {seed_payload.get('claim')}")

def hits_to_frame(hits):
    """Convert Qdrant search hits into a compact display table."""
    rows = []
    for hit in hits:
        payload = hit.payload or {}
        rows.append(
            {
                "score": hit.score,
                "pair_id": payload.get("pair_id"),
                "split": payload.get("split"),
                "label": payload.get("label"),
                "claim_type": payload.get("claim_type"),
                "claim": payload.get("claim"),
            }
        )
    return pd.DataFrame(rows)

for vector_name in ["pair_text_vec", "claim_vec", "image_vec"]:
    if vector_name not in seed_vectors:
        print(f"Skipping {vector_name}: this collection does not have that vector on the seed point.")
        continue

    # `using` selects which named vector Qdrant should compare against.
    hits = query_points(
        client,
        COLLECTION,
        vector_name=vector_name,
        vector=seed_vectors[vector_name],
        limit=5,
    )
    print(f"\nNearest neighbors by {vector_name}")
    display(hits_to_frame(hits))

Seed record
  pair_id: pair:val:2410.01727v1:631:e591e8dfb5724723:231df64e377a39cd
  label:   entailed
  claim:   On the Eedi dataset, sparseKT model’s AUC increases by 3.98 points (from 74.98 to 78.96), representing a relative gain of 5.31%, the largest percentage improvement among all models.

Nearest neighbors by pair_text_vec


,score,pair_id,split,label,claim_type,claim
0,1.000000,pair:val:2410.01727v1:631:e591e8dfb5724723:231...,val,entailed,direct,"On the Eedi dataset, sparseKT model’s AUC incr..."
1,0.682211,pair:test:2411.15223v1:11:205db494f9af6551:f39...,test,entailed,direct,The Paper Model's AUC (0.7850) exceeds DeepFM'...
2,0.617438,pair:test:2409.12479v1:116:8d4a178d7e747bc7:1d...,test,entailed,direct,The average AUC across all OOD datasets increa...
3,0.602161,pair:test:2409.17504v1:756:f433c46ba56b9d90:11...,test,refuted,direct,"As k increases from 1 to 6, AUROC on TruthfulQ..."
4,0.566189,pair:test:2410.10177v1:338:a6c8a1c42fef3b36:67...,test,entailed,analytical,"In latent diffusion models, AUC-ROC increases ..."



Nearest neighbors by claim_vec


,score,pair_id,split,label,claim_type,claim
0,1.000000,pair:val:2410.01727v1:631:e591e8dfb5724723:231...,val,entailed,direct,"On the Eedi dataset, sparseKT model’s AUC incr..."
1,0.664996,pair:test:2411.15223v1:11:205db494f9af6551:f39...,test,entailed,direct,The Paper Model's AUC (0.7850) exceeds DeepFM'...
2,0.629371,pair:test:2409.12479v1:116:8d4a178d7e747bc7:1d...,test,entailed,direct,The average AUC across all OOD datasets increa...
3,0.607827,pair:test:2409.17504v1:756:f433c46ba56b9d90:11...,test,refuted,direct,"As k increases from 1 to 6, AUROC on TruthfulQ..."
4,0.564994,pair:val:2409.17608v1:409:5b5107064045049c:377...,val,refuted,direct,"On the ShanghaiTech (SHT) dataset, our flow-ba..."



Nearest neighbors by image_vec


,score,pair_id,split,label,claim_type,claim
0,1.000000,pair:val:2410.01727v1:631:e591e8dfb5724723:231...,val,entailed,direct,"On the Eedi dataset, sparseKT model’s AUC incr..."
1,0.843438,pair:test:2410.01485v1:231:c65deeda5d04642f:3f...,test,refuted,direct,Using all full layers achieves an average perf...
2,0.843438,pair:test:2410.01485v1:235:3f97239ec9dc2a6f:3f...,test,entailed,analytical,Strategically embedding 12 full attention laye...
3,0.840936,pair:test:2410.23910v1:305:b1726da4a56a029a:0b...,test,refuted,direct,Our method's average ROC-AUC of 0.6235 exceeds...
4,0.836240,pair:val:2411.12115v1:687:f8e73c6cdbea9356:a83...,val,entailed,direct,Pruned LD3M with DM achieves a 3.3 percentage-...


## 5. Search with Metadata Filters

Filters are useful for split-aware experiments and controlled retrieval. The example below searches only validation records that are entailed chart pairs.

In [5]:
filter_val_entailed_chart = models.Filter(
    must=[
        models.FieldCondition(key="split", match=models.MatchValue(value="val")),
        models.FieldCondition(key="label", match=models.MatchValue(value="entailed")),
        models.FieldCondition(key="modality", match=models.MatchValue(value="chart")),
    ]
)

# Reuse the seed's pair text vector, but restrict the candidate set by payload metadata.
filtered_hits = query_points(
    client,
    COLLECTION,
    vector_name="pair_text_vec",
    vector=seed_vectors["pair_text_vec"],
    limit=5,
    query_filter=filter_val_entailed_chart,
)
hits_to_frame(filtered_hits)

,score,pair_id,split,label,claim_type,claim
0,0.398556,pair:val:2411.06866v1:660:9e9b4de21a51ca99:150...,val,entailed,direct,"OpenBookQA accuracy peaks at 72.3% when n=100,..."
1,0.395314,pair:val:2409.10343v1:48:bc171dd1a2e5dde9:b6a9...,val,entailed,direct,"Recall@10 peaks at 0.10 when ε^max is 3%, risi..."
2,0.387149,pair:val:2411.00049v1:104:663dbf6f7ac32a63:431...,val,entailed,direct,"Between threshold t=0 and t=0.9, Hatespeech ac..."
3,0.361186,pair:val:2410.03705v2:324:ff11c4d47eca7304:ea5...,val,entailed,analytical,"On the EEG Eye State dataset, ensemble-based G..."
4,0.352450,pair:val:2411.15893v1:1196:da33bfe0b9c5becb:98...,val,entailed,direct,Increasing the number of dimensions from 0 to ...


## 6. Retrieve by Deterministic Point ID

The payload stores a human-readable `pair_id`. Qdrant point IDs are deterministic UUIDv5 values derived from that `pair_id`, so a record can be retrieved exactly when the human-readable ID is known.

In [6]:
pair_id = seed_payload["pair_id"]
qdrant_id = point_uuid(pair_id)

# Retrieve returns exact point IDs, unlike nearest-neighbor search which ranks by vector similarity.
retrieved = client.retrieve(
    collection_name=COLLECTION,
    ids=[qdrant_id],
    with_payload=True,
    with_vectors=False,
)

print(f"pair_id:   {pair_id}")
print(f"qdrant_id: {qdrant_id}")
print(f"retrieved: {len(retrieved)} point(s)")
retrieved[0].payload if retrieved else None

pair_id:   pair:val:2410.01727v1:631:e591e8dfb5724723:231df64e377a39cd
qdrant_id: 00463a2d-9d9e-572c-aef1-7d0c232d0802
retrieved: 1 point(s)


{'pair_id': 'pair:val:2410.01727v1:631:e591e8dfb5724723:231df64e377a39cd',
 'claim_id': 'claim:val:2410.01727v1:631:e591e8dfb5724723',
 'evidence_item_id': 'evidence:2410.01727v1:table:direct:f27058057a68b9e2',
 'paperid': '2410.01727v1',
 'request_id': '631',
 'split': 'val',
 'claim': 'On the Eedi dataset, sparseKT model’s AUC increases by 3.98 points (from 74.98 to 78.96), representing a relative gain of 5.31%, the largest percentage improvement among all models.',
 'label': 'entailed',
 'label_id': 1,
 'claim_type': 'direct',
 'modality': 'table',
 'image_path': '/Users/conglongxu/Projects/Erdos-2016-Summer/datasets/SciVer/datasets--chengyewang--SciVer/blobs/5ba47aeb27606404a5e782133672bf22f86b8f31d6dff9806884ee277a93dfd7',
 'section_json': '["5.1"]',
 'n_evidence_items': 1,
 'gold_pair': True,
 'pair_mode': 'single_visual_only',
 'feature_version': 'sciver_vector_db_v1',
 'embedding_model_text': 'sentence-transformers/all-MiniLM-L6-v2',
 'embedding_model_image': 'openai/clip-vit-b